# 13 — Final Signer-Independent Evaluation and Statistical Inference

**Goal.** Answer the thesis questions from the frozen predictions of 10–12. This is analysis only: nothing is retrained and no hyperparameter exists here.

**Question hierarchy**
- **RQ1 (primary).** How well do RF, BiLSTM and BiGRU recognise signs from **completely unseen signers**?
- **RQ2.** Does Reduced (92 face landmarks) preserve or improve accuracy vs Full (468), and what does it save computationally?
- **RQ3.** Is the Full→Reduced difference consistent across folds and individual signers?
- **Diagnostic.** What happens when signer overlap is allowed (control split)?

All six model × representation configurations are reported. The best one is identified from recomputed values and was not pre-designated.

**Metric hierarchy.** Primary: pooled OOF **accuracy**. Key secondary: macro-F1 and signer-macro accuracy. Then min/max/spread over the 16 signers, weighted-F1, balanced accuracy, top-5, per-fold results, overlap gaps, efficiency, per-class recall and confusion matrices.

**Verification before analysis.** All upstream contract IDs and 12 prediction digests are recomputed from the saved arrays. A signer-independence audit (16 signers, 8 disjoint 12/2/2 folds, each signer tested once, test folds tiling all 6,000 clips, every OOF prediction from a fold that held out that clip's signer) must pass, or the notebook aborts.

**Frozen statistical plan (RQ2)**
1. `d_k = acc(Reduced, k) − acc(Full, k)` for the 8 folds, with `T = mean(d)`.
2. Exact two-sided paired **sign-flip test** over all 2⁸ = 256 sign assignments. The minimum attainable p is 2/256.
3. Accuracy only. **Holm** correction across the 3 model-specific tests.
4. **Fold-clustered bootstrap** (10,000 resamples, seed 45000) for 95% intervals, as a sensitivity analysis.
5. With n = 8 the p-value has a hard floor, so **effect sizes and intervals lead, and p-values are reported alongside**. No equivalence claim and no hard-coded expected values.

**No cross-model significance test.** It was not pre-specified, and RF (physical-time engineered summaries) and the RNNs (native-frame sequences) differ in both architecture and time base. The comparison stays descriptive.

**Wording kept throughout.** *Signer-independent evaluation* means the model never saw the test signer. It does **not** mean the representation contains no signer information; the overlap control shows that such cues exist.

**Inputs → Outputs.** 07, 10, 11, 12 → `final_primary_signer_independent.csv`, `final_secondary_representation.csv`, `final_secondary_inference.csv`, `final_fold_metrics.csv`, `final_signer_metrics.csv`, `final_control_gap.csv`, `final_efficiency.csv`, class-level tables, confusion matrices, `final_analysis_contract.json`, `final_validation_report.json`.

In [ ]:
# ============================================================
# CELL 1 — Configuration and the frozen analysis policy
# ============================================================
from __future__ import annotations

import hashlib
import itertools
import json
import os
import platform
import sys
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             confusion_matrix, f1_score, recall_score)

NOTEBOOK = "nb13"
ANALYSIS_POLICY_VERSION = "nb13-v1"

# ---- frozen upstream provenance --------------------------------------------
REQUIRED_EXPERIMENT_ID = "1308155924042fc2"
REQUIRED_LOADER_CONTRACT_ID = "112c4e5a438540d0"
REQUIRED_STANDARDISER_SET_ID = "0cf4c790ec474110"
REQUIRED_TIME_METADATA_ID = "e30a51110d2f9324"
REQUIRED_FEATURE_CONTRACT_ID = "bfcce8a5fa013fab"
REQUIRED_FILTER_SET_ID = "3b5f7a8bfc08656c"

# Notebook 7 table hashes, re-verified here so the fold definitions are trusted.
REQUIRED_NB7_HASHES = {
    "manifest": "0bc51c943ed2e688",
    "fold_assignments": "bc94d5ba486ff49c",
    "control_split": "fcc419d92e546470",
    "label_map": "83ac9fcbf7c01a59",
}

# ---- the three frozen model families ---------------------------------------
MODELS: Tuple[str, ...] = ("RF", "BiLSTM", "BiGRU")
BRANCHES: Tuple[str, ...] = ("full", "reduced")
MODEL_SPEC = {
    "RF": {"prefix": "rf", "manifest": "rf_model_manifest.json",
           "contract_key": "rf_model_contract_id", "contract": "fc10c99099c63934",
           "contract_file": "rf_model_contract.json",
           "report": "rf_validation_report.json",
           "time_base": "engineered per-clip summaries in PHYSICAL TIME (motion scaled "
                        "by each clip's frame rate)"},
    "BiLSTM": {"prefix": "nn", "manifest": "nn_model_manifest.json",
               "contract_key": "nn_model_contract_id", "contract": "95d9c6457ab2ab14",
               "contract_file": "nn_model_contract.json",
               "report": "nn_validation_report.json",
               "time_base": "ordered NATIVE-FRAME sequences"},
    "BiGRU": {"prefix": "bigru", "manifest": "bigru_model_manifest.json",
              "contract_key": "bigru_model_contract_id", "contract": "cc4595f6f33e0658",
              "contract_file": "bigru_model_contract.json",
              "report": "bigru_validation_report.json",
              "time_base": "ordered NATIVE-FRAME sequences"},
}
REQUIRED_OOF_DIGEST = {
    ("RF", "full"): "f9a3bb8d8613a675", ("RF", "reduced"): "74711b561b772bec",
    ("BiLSTM", "full"): "3e312fc23b949766", ("BiLSTM", "reduced"): "822a9aee1bd97ba4",
    ("BiGRU", "full"): "1d032beb2a8488f9", ("BiGRU", "reduced"): "a1596bc81ef2a25a",
}
REQUIRED_CONTROL_DIGEST = {
    ("RF", "full"): "59882aeb7985ed99", ("RF", "reduced"): "84aa272794d73def",
    ("BiLSTM", "full"): "636f7e30ad419e2e", ("BiLSTM", "reduced"): "eb29e13406bc94ba",
    ("BiGRU", "full"): "c5d4fa56a90dc491", ("BiGRU", "reduced"): "f0a18b51d027bb46",
}

N_CLIPS, N_LABELS, N_SIGNERS, N_FOLDS = 6000, 200, 16, 8
TRAIN_SIGNERS_PER_FOLD = 12
CONTROL_SPLIT_SIZES = {"train": 4500, "validation": 750, "test": 750}
TOP_K = 5

# ---- frozen statistical plan for the SECONDARY representation question ------
PRIMARY_METRIC = "accuracy"                 # single primary headline metric
KEY_SECONDARY_METRICS = ("macro_f1", "signer_macro_accuracy")
SECONDARY_METRICS = ("weighted_f1", "balanced_accuracy", f"top{TOP_K}_accuracy")
INFERENCE_METRIC = "accuracy"               # the only inferential metric
SIGN_FLIP_ASSIGNMENTS = 2 ** N_FOLDS        # 256
SIGN_FLIP_MIN_P = 2.0 / SIGN_FLIP_ASSIGNMENTS
BOOTSTRAP_RESAMPLES = 10_000
BOOTSTRAP_SEED = 45000
BOOTSTRAP_CI = 0.95
BOOTSTRAP_UNIT = "folds resampled as clusters"
MULTIPLICITY = "Holm step-down across the three model-specific accuracy tests (m = 3)"
CLASS_RECALL_THRESHOLD = 0.50                # for the concise per-class summary
CLASS_SUMMARY_TOP_N = 10

SEARCH_ROOT = "/kaggle/input"
WORKING = Path("/kaggle/working")
WORKING.mkdir(parents=True, exist_ok=True)

OUT = {
    "contract": WORKING / "final_analysis_contract.json",
    "manifest": WORKING / "final_analysis_manifest.json",
    "primary": WORKING / "final_primary_signer_independent.csv",
    "fold_metrics": WORKING / "final_fold_metrics.csv",
    "signer_metrics": WORKING / "final_signer_metrics.csv",
    "model_family": WORKING / "final_model_family_comparison.csv",
    "secondary": WORKING / "final_secondary_representation.csv",
    "secondary_inference": WORKING / "final_secondary_inference.csv",
    "control_gap": WORKING / "final_control_gap.csv",
    "efficiency": WORKING / "final_efficiency.csv",
    "class_summary": WORKING / "final_class_summary.csv",
    "class_recall": WORKING / "final_class_recall_supplementary.csv",
    "limitations": WORKING / "final_limitations.json",
    "thesis_tables": WORKING / "final_thesis_tables.md",
    "report_json": WORKING / "final_validation_report.json",
    "report_csv": WORKING / "final_validation_report.csv",
}

ENVIRONMENT = {
    "notebook": NOTEBOOK, "analysis_policy_version": ANALYSIS_POLICY_VERSION,
    "python": platform.python_version(), "numpy": np.__version__,
    "pandas": pd.__version__,
}

STARTED = time.time()
CHECKS: List[Dict[str, Any]] = []


def record(name: str, passed: bool, detail: Any = "", status: str = "") -> bool:
    passed = bool(passed)
    status = status or ("pass" if passed else "fail")
    CHECKS.append({"check": name, "passed": passed, "status": status,
                   "detail": str(detail)})
    label = {"pass": "PASS", "fail": "FAIL", "skip": "SKIP"}[status]
    print(f"  {label}  {name:<64} {detail}")
    return passed


def require(name: str, passed: bool, detail: Any = "") -> None:
    if not record(name, passed, detail):
        raise RuntimeError(f"{name} failed ({detail}). Notebook 13 refuses to analyse or "
                           "publish on top of a failed check.")


def skip(name: str, reason: Any) -> None:
    record(name, True, reason, status="skip")


def failures() -> List[Dict[str, Any]]:
    return [entry for entry in CHECKS if entry["status"] == "fail"]


def sha16(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]


def array_digest(*arrays: np.ndarray) -> str:
    """The recipe used by notebooks 10, 11 and 12, so digests are comparable."""
    digest = hashlib.sha256()
    for array in arrays:
        contiguous = np.ascontiguousarray(array)
        digest.update(str(contiguous.dtype).encode("utf-8"))
        digest.update(str(contiguous.shape).encode("utf-8"))
        digest.update(contiguous.tobytes())
    return digest.hexdigest()[:16]


def canonical_hash(frame: pd.DataFrame, columns: Sequence[str]) -> str:
    """The notebook 7 recipe, reimplemented here so notebook 8 need not be attached."""
    ordered = frame.sort_values(list(columns), kind="mergesort")[list(columns)]
    lines = ["\t".join(f"{column}={record_[column]}" for column in columns)
             for record_ in ordered.to_dict("records")]
    return hashlib.sha256(("\n".join(lines) + "\n").encode("utf-8")).hexdigest()[:16]


def atomic_write(path: Path, writer) -> None:
    temporary = path.parent / (path.name + ".tmp")
    with open(temporary, "wb") as handle:
        writer(handle)
    os.replace(temporary, path)


print("Environment")
for key, value in ENVIRONMENT.items():
    print(f"  {key}: {value}")

print("\nTHESIS HIERARCHY")
print("  PRIMARY    : signer-independent recognition of Bangla signs by unseen signers,")
print("               across Random Forest, BiLSTM and BiGRU.")
print("  SECONDARY  : does the Reduced-face representation preserve or improve")
print("               signer-independent recognition, and what does it save?")
print("  DIAGNOSTIC : what happens when signer overlap is intentionally allowed?")
print("  All six model x representation configurations are reported. The best observed")
print("  configuration is identified from the recomputed values, not pre-designated.")

print("\nMETRIC HIERARCHY")
print(f"  primary headline : pooled signer-independent {PRIMARY_METRIC}")
print(f"  key secondary    : {', '.join(KEY_SECONDARY_METRICS)}")
print("  prominent descr. : minimum / maximum signer accuracy, signer spread, all 16")
print(f"  secondary        : {', '.join(SECONDARY_METRICS)}")
print("  supplementary    : per-class recall, 200x200 confusion matrices")

print("\nFROZEN STATISTICAL PLAN (secondary representation question)")
print(f"  statistic     : mean of the {N_FOLDS} paired Reduced-minus-Full fold "
      f"{INFERENCE_METRIC} differences")
print(f"  test          : exhaustive two-sided paired sign-flip over "
      f"{SIGN_FLIP_ASSIGNMENTS} assignments")
print(f"  minimum p     : {SIGN_FLIP_MIN_P:.7f}")
print(f"  multiplicity  : {MULTIPLICITY}; Holm needs no independence assumption")
print(f"  sensitivity   : {BOOTSTRAP_RESAMPLES:,}-resample bootstrap, seed "
      f"{BOOTSTRAP_SEED}, {BOOTSTRAP_UNIT}, {BOOTSTRAP_CI:.0%} percentile")
print(f"  inferential metric: {INFERENCE_METRIC} ONLY. Macro-F1 gets no test family.")
print("  no equivalence claim; no cross-model significance test; nothing hard-coded.")
print("\n  Reporting rule: with n = 8 the permutation p has a hard floor and cannot")
print("  distinguish a small effect from a large one, so effect sizes, per-fold ranges")
print("  and the bootstrap interval lead. p-values are reported alongside.")

In [ ]:
# ============================================================
# CELL 2 — Provenance: re-hash notebook 7, re-verify every model contract and digest
# ============================================================

print("LOCATING FROZEN ARTEFACTS")


def find_one(pattern: str, description: str, root: str = SEARCH_ROOT) -> Path:
    hits = sorted(Path(root).glob(pattern))
    if not hits:
        raise FileNotFoundError(f"No {description} found under {root} "
                                f"(pattern {pattern!r}). Attach the producing notebook "
                                "output.")
    if len(hits) > 1:
        listing = "\n".join(f"    {h}" for h in hits)
        raise RuntimeError(f"{len(hits)} candidates for {description}; refusing to "
                           f"choose.\n{listing}")
    return hits[0]


NB7_DIR = find_one("**/experiment_manifest.json", "notebook 7 experiment manifest").parent
MODEL_DIR: Dict[str, Path] = {}
for model, spec in MODEL_SPEC.items():
    MODEL_DIR[model] = find_one(f"**/{spec['manifest']}", f"{model} manifest").parent
    print(f"  {model:<7}: {MODEL_DIR[model]}")
print(f"  NB7    : {NB7_DIR}")

# ---- the frozen notebook 7 tables ------------------------------------------
print("\nNOTEBOOK 7 TABLES (re-hashed here, so the fold definitions are trusted)")
MANIFEST_COLUMNS = ["clip_id", "label", "label_id", "signer_id", "session",
                    "relative_path", "original_frames"]
FOLD_COLUMNS = ["fold", "clip_id", "split"]
CONTROL_COLUMNS = ["clip_id", "split"]
TEXT_COLUMNS = {
    "master_manifest.csv": ["clip_id", "label", "signer_id", "session", "relative_path"],
    "fold_assignments.csv": ["clip_id", "signer_id", "split"],
    "CONTROL_signer_overlap_split.csv": ["clip_id", "signer_id", "label", "split",
                                         "split_type"],
}


def read_frozen_csv(name: str, columns: Sequence[str], expected: str) -> pd.DataFrame:
    """Two read strategies; both parse the same bytes and the hash must still match."""
    attempts = []
    for label, kwargs in (("text-preserving",
                           {"dtype": {c: str for c in TEXT_COLUMNS[name]},
                            "keep_default_na": False}),
                          ("pandas-default", {})):
        frame = pd.read_csv(NB7_DIR / name, **kwargs)
        digest = canonical_hash(frame, columns)
        attempts.append((label, digest))
        if digest == expected:
            print(f"  {name:<36} {digest}  ({label})")
            return frame
    detail = "; ".join(f"{a} -> {d}" for a, d in attempts)
    raise RuntimeError(f"{name} does not reproduce its frozen hash {expected}. {detail}")


manifest_table = read_frozen_csv("master_manifest.csv", MANIFEST_COLUMNS,
                                 REQUIRED_NB7_HASHES["manifest"])
fold_table = read_frozen_csv("fold_assignments.csv", FOLD_COLUMNS,
                             REQUIRED_NB7_HASHES["fold_assignments"])
control_table = read_frozen_csv("CONTROL_signer_overlap_split.csv", CONTROL_COLUMNS,
                                REQUIRED_NB7_HASHES["control_split"])
label_map = json.loads((NB7_DIR / "label_map.json").read_text(encoding="utf-8"))
LABEL_TO_ID = {str(k): int(v) for k, v in label_map["label_to_id"].items()}
label_hash = sha16(json.dumps(LABEL_TO_ID, sort_keys=True, separators=(",", ":")))
print(f"  {'label_map.json':<36} {label_hash}")
require("all four notebook 7 artefacts reproduce their frozen hashes",
        label_hash == REQUIRED_NB7_HASHES["label_map"], "manifest, folds, control, labels")

for column in ("clip_id", "signer_id", "split"):
    fold_table[column] = fold_table[column].astype(str)
for column in ("clip_id", "split"):
    control_table[column] = control_table[column].astype(str)
manifest_table["clip_id"] = manifest_table["clip_id"].astype(str)
manifest_table["signer_id"] = manifest_table["signer_id"].astype(str)

CLIP_IDS = sorted(manifest_table["clip_id"].tolist())
LABEL_OF = manifest_table.set_index("clip_id")["label_id"].astype(int)
SIGNER_OF = manifest_table.set_index("clip_id")["signer_id"].astype(str)
ID_TO_LABEL = {v: k for k, v in LABEL_TO_ID.items()}
Y_TRUE = LABEL_OF.loc[CLIP_IDS].to_numpy(dtype=np.int64)
SIGNERS = SIGNER_OF.loc[CLIP_IDS].to_numpy().astype(str)
SIGNER_ORDER = sorted(set(SIGNERS.tolist()), key=lambda s: int(str(s).lstrip("p")))
require("the frozen manifest holds 6,000 clips, 200 labels, 16 signers",
        len(manifest_table) == N_CLIPS and manifest_table["label_id"].nunique() == N_LABELS
        and manifest_table["signer_id"].nunique() == N_SIGNERS,
        f"{len(manifest_table):,} / {manifest_table['label_id'].nunique()} / "
        f"{manifest_table['signer_id'].nunique()}")

# ---- model contracts and reports -------------------------------------------
# Each of notebooks 10, 11 and 12 hashed its contract with
#   sha256(json.dumps(contract, sort_keys=True, separators=(",", ":"), default=str))[:16]
# and saved the contract object both standalone and nested in its manifest. That recipe
# is reapplied here, so the ids are INDEPENDENTLY RECOMPUTED from the saved objects
# rather than merely read and compared.
def recompute_contract_id(contract_object: Dict[str, Any]) -> str:
    return sha16(json.dumps(contract_object, sort_keys=True, separators=(",", ":"),
                            default=str))


print("\nMODEL CONTRACTS — INDEPENDENTLY RECOMPUTED FROM THE SAVED CONTRACT OBJECTS")
MANIFESTS: Dict[str, Dict[str, Any]] = {}
for model, spec in MODEL_SPEC.items():
    MANIFESTS[model] = json.loads(
        (MODEL_DIR[model] / spec["manifest"]).read_text(encoding="utf-8"))
    recorded = str(MANIFESTS[model][spec["contract_key"]])
    nested_key = spec["contract_key"].replace("_id", "")
    require(f"{model} manifest carries its contract object under {nested_key!r}",
            nested_key in MANIFESTS[model], "available for recomputation")
    from_manifest = recompute_contract_id(MANIFESTS[model][nested_key])
    print(f"  {model:<7} recorded {recorded}  recomputed {from_manifest}")
    require(f"{model} contract id recomputed from the nested contract object",
            from_manifest == recorded == spec["contract"], from_manifest)

    contract_path = MODEL_DIR[model] / spec["contract_file"]
    if contract_path.exists():
        from_file = recompute_contract_id(
            json.loads(contract_path.read_text(encoding="utf-8")))
        require(f"{model} contract id recomputed from {spec['contract_file']}",
                from_file == spec["contract"], from_file)
    else:
        skip(f"{model} contract id recomputed from {spec['contract_file']}",
             "standalone contract file not attached; the nested object was used")
    report = json.loads((MODEL_DIR[model] / spec["report"]).read_text(encoding="utf-8"))
    require(f"{model} reported a clean run",
            report["verdict"] == "PASS" and report["checks_failed"] == 0,
            f"{report['checks_passed']}/{report['checks_total']} passed")
    upstream = MANIFESTS[model]["upstream"]
    require(f"{model} records the frozen experiment id",
            str(upstream["experiment_id"]) == REQUIRED_EXPERIMENT_ID,
            upstream["experiment_id"])
    require(f"{model} records the frozen loader contract id",
            str(upstream["loader_contract_id"]) == REQUIRED_LOADER_CONTRACT_ID,
            upstream["loader_contract_id"])

# the neural families additionally carry the standardiser set and time metadata ids
for model in ("BiLSTM", "BiGRU"):
    upstream = MANIFESTS[model]["upstream"]
    require(f"{model} records the frozen standardiser set id",
            str(upstream["standardiser_set_id"]) == REQUIRED_STANDARDISER_SET_ID,
            upstream["standardiser_set_id"])
    require(f"{model} records the frozen time metadata id",
            str(upstream["time_metadata_id"]) == REQUIRED_TIME_METADATA_ID,
            upstream["time_metadata_id"])
rf_upstream = MANIFESTS["RF"]["upstream"]
require("RF records the frozen feature contract id",
        str(rf_upstream["feature_contract_id"]) == REQUIRED_FEATURE_CONTRACT_ID,
        rf_upstream["feature_contract_id"])
require("RF records the frozen filter set id",
        str(rf_upstream["filter_set_id"]) == REQUIRED_FILTER_SET_ID,
        rf_upstream["filter_set_id"])
require("all three models share one experiment id",
        len({str(MANIFESTS[m]["upstream"]["experiment_id"]) for m in MODELS}) == 1,
        REQUIRED_EXPERIMENT_ID)

# ---- prediction digests, recomputed from the arrays ------------------------
print("\nPREDICTION DIGESTS (recomputed from the saved arrays, not read)")
OOF: Dict[Tuple[str, str], Dict[str, np.ndarray]] = {}
CONTROL: Dict[Tuple[str, str], Dict[str, np.ndarray]] = {}
for model in MODELS:
    prefix = MODEL_SPEC[model]["prefix"]
    for branch in BRANCHES:
        with np.load(MODEL_DIR[model] / f"{prefix}_oof_{branch}.npz",
                     allow_pickle=False) as archive:
            block = {key: np.array(archive[key]) for key in archive.files
                     if key != "__meta__"}
        block["clip_id"] = block["clip_id"].astype(str)
        block["signer_id"] = block["signer_id"].astype(str)
        OOF[(model, branch)] = block
        digest = array_digest(block["y_pred"], block["proba"])
        require(f"{model} {branch} out-of-fold digest reproduces",
                digest == REQUIRED_OOF_DIGEST[(model, branch)], digest)

        with np.load(MODEL_DIR[model] / f"{prefix}_control_{branch}.npz",
                     allow_pickle=False) as archive:
            block = {key: np.array(archive[key]) for key in archive.files
                     if key != "__meta__"}
        block["clip_id"] = block["clip_id"].astype(str)
        CONTROL[(model, branch)] = block
        digest = array_digest(block["y_pred"], block["proba"])
        require(f"{model} {branch} control digest reproduces",
                digest == REQUIRED_CONTROL_DIGEST[(model, branch)], digest)
print(f"\n  {len(OOF)} out-of-fold and {len(CONTROL)} control prediction sets verified")

In [ ]:
# ============================================================
# CELL 3 — Signer-independence audit, from the artefacts rather than from claims
# ============================================================
# This is the evidence base for the PRIMARY thesis claim. Every property below is
# derived from the frozen notebook 7 tables and the saved predictions.

print("FOLD STRUCTURE (rebuilt from fold_assignments.csv)")
FOLDS: Dict[int, Dict[str, Tuple[str, ...]]] = {}
for fold in sorted(int(f) for f in fold_table["fold"].unique()):
    subset = fold_table[fold_table["fold"] == fold]
    FOLDS[fold] = {split: tuple(sorted(
        subset.loc[subset["split"] == split, "signer_id"].unique(),
        key=lambda s: int(str(s).lstrip("p")))) for split in ("train", "validation", "test")}
print(f"{'fold':>4}  {'test':<10} {'validation':<10} {'train signers':<44} clips T/V/Te")
for fold in range(N_FOLDS):
    definition = FOLDS[fold]
    subset = fold_table[fold_table["fold"] == fold]
    sizes = [int((subset["split"] == s).sum()) for s in ("train", "validation", "test")]
    print(f"{fold:>4}  {' '.join(definition['test']):<10} "
          f"{' '.join(definition['validation']):<10} "
          f"{' '.join(definition['train']):<44} {sizes[0]}/{sizes[1]}/{sizes[2]}")

require("16 signers across 8 folds",
        len(FOLDS) == N_FOLDS
        and len(set(fold_table["signer_id"])) == N_SIGNERS, f"{N_FOLDS} / {N_SIGNERS}")
require("every fold has 12 train / 2 validation / 2 test signers",
        all(len(FOLDS[f]["train"]) == TRAIN_SIGNERS_PER_FOLD
            and len(FOLDS[f]["validation"]) == 2 and len(FOLDS[f]["test"]) == 2
            for f in FOLDS), "8 of 8")
require("train / validation / test signer sets disjoint in every fold",
        all(not (set(FOLDS[f]["train"]) & set(FOLDS[f]["validation"]))
            and not (set(FOLDS[f]["train"]) & set(FOLDS[f]["test"]))
            and not (set(FOLDS[f]["validation"]) & set(FOLDS[f]["test"]))
            for f in FOLDS), "8 of 8")

test_counts: Dict[str, int] = {}
validation_counts: Dict[str, int] = {}
for fold in range(N_FOLDS):
    for signer in FOLDS[fold]["test"]:
        test_counts[signer] = test_counts.get(signer, 0) + 1
    for signer in FOLDS[fold]["validation"]:
        validation_counts[signer] = validation_counts.get(signer, 0) + 1
require("every signer is a test signer exactly once",
        len(test_counts) == N_SIGNERS and set(test_counts.values()) == {1},
        f"{len(test_counts)} signers")
require("every signer is a validation signer exactly once",
        len(validation_counts) == N_SIGNERS and set(validation_counts.values()) == {1},
        f"{len(validation_counts)} signers")

pooled_test_clips: List[str] = []
for fold in range(N_FOLDS):
    subset = fold_table[(fold_table["fold"] == fold) & (fold_table["split"] == "test")]
    pooled_test_clips.extend(subset["clip_id"].tolist())
require("the 8 test folds together cover all 6,000 clips exactly once",
        len(pooled_test_clips) == N_CLIPS
        and sorted(pooled_test_clips) == CLIP_IDS, f"{len(pooled_test_clips):,} rows, "
        f"{len(set(pooled_test_clips)):,} unique")

# ---- the decisive property, checked on every saved prediction --------------
print("\nSIGNER INDEPENDENCE OF EVERY SAVED PREDICTION")
for (model, branch), block in OOF.items():
    require(f"{model} {branch}: 6,000 rows covering every clip exactly once",
            block["clip_id"].size == N_CLIPS
            and sorted(block["clip_id"].tolist()) == CLIP_IDS, "no gaps, no duplicates")
    order = np.argsort(block["clip_id"], kind="mergesort")
    for key in ("clip_id", "signer_id", "label_id", "fold", "y_true", "y_pred"):
        block[key] = block[key][order]
    block["proba"] = block["proba"][order]
    require(f"{model} {branch}: labels match the frozen manifest",
            np.array_equal(block["y_true"].astype(np.int64), Y_TRUE), "aligned")
    require(f"{model} {branch}: signers match the frozen manifest",
            np.array_equal(block["signer_id"], SIGNERS), "aligned")
    held_out = all(block["signer_id"][i] in FOLDS[int(block["fold"][i])]["test"]
                   for i in range(N_CLIPS))
    require(f"{model} {branch}: EVERY prediction came from a fold that held out its "
            "signer", held_out, "signer-independent by construction")
    not_trained = all(block["signer_id"][i] not in FOLDS[int(block["fold"][i])]["train"]
                      and block["signer_id"][i]
                      not in FOLDS[int(block["fold"][i])]["validation"]
                      for i in range(N_CLIPS))
    require(f"{model} {branch}: no prediction came from a model that trained or "
            "validated on that signer", not_trained, "no train or validation exposure")
    require(f"{model} {branch}: probabilities are finite and normalised",
            bool(np.isfinite(block["proba"]).all())
            and float(np.abs(block["proba"].sum(axis=1) - 1.0).max()) < 1e-3,
            f"max |rowsum-1| {float(np.abs(block['proba'].sum(axis=1) - 1.0).max()):.2e}")
    require(f"{model} {branch}: predictions equal argmax of the probabilities",
            np.array_equal(block["y_pred"], np.argmax(block["proba"], axis=1)),
            "class order 0..199")

require("all six prediction sets cover the same clips in the same order",
        all(np.array_equal(OOF[(m, b)]["clip_id"], CLIP_IDS)
            for m in MODELS for b in BRANCHES), "paired clip by clip")

# ---- the control is separate by construction ------------------------------
print("\nINTENTIONAL SIGNER-OVERLAP CONTROL (kept separate)")
control_test_clips = sorted(control_table.loc[control_table["split"] == "test", "clip_id"])
control_train_signers = set(control_table.loc[control_table["split"] == "train",
                                              "signer_id"].astype(str))
control_test_signers = set(control_table.loc[control_table["split"] == "test",
                                             "signer_id"].astype(str))
shared = control_train_signers & control_test_signers
print(f"  control splits          : "
      f"{ {s: int((control_table['split'] == s).sum()) for s in ('train','validation','test')} }")
print(f"  signers shared train/test: {len(shared)} of {N_SIGNERS} (intentional)")
require("control split sizes are the frozen 4500 / 750 / 750",
        {s: int((control_table["split"] == s).sum())
         for s in ("train", "validation", "test")} == CONTROL_SPLIT_SIZES,
        CONTROL_SPLIT_SIZES)
require("the control shares all 16 signers between train and test by design",
        len(shared) == N_SIGNERS, f"{len(shared)} of {N_SIGNERS}")
for (model, branch), block in CONTROL.items():
    require(f"{model} {branch}: control holds the frozen 750 test clips",
            block["y_true"].size == CONTROL_SPLIT_SIZES["test"]
            and sorted(block["clip_id"].tolist()) == control_test_clips,
            f"{block['y_true'].size}")
require("no control clip set was pooled into any out-of-fold array",
        all(OOF[(m, b)]["clip_id"].size == N_CLIPS for m in MODELS for b in BRANCHES),
        "out-of-fold arrays hold exactly the 6,000 signer-independent predictions")

print("\n  Signer-independent EVALUATION is established by the protocol above: every")
print("  reported prediction comes from a model that never trained or validated on that")
print("  clip's signer. This does NOT establish that the representation contains no")
print("  signer information, and that stronger claim is not made anywhere in this")
print("  notebook. The overlap control below shows signer-associated cues are present")
print("  and usable, which is exactly why the signer-independent protocol is required.")

In [ ]:
# ============================================================
# CELL 4 — PRIMARY: signer-independent recognition of unseen signers
# ============================================================
# Every number below is recomputed from the saved predictions and then cross-checked
# against the upstream per-model CSVs. Nothing is taken on trust and nothing is
# hard-coded.


def top_k_accuracy(proba: np.ndarray, y_true: np.ndarray, k: int = TOP_K) -> float:
    if proba.shape[1] <= k:
        return 1.0
    top = np.argpartition(-proba, kth=k - 1, axis=1)[:, :k]
    return float(np.mean([y_true[i] in top[i] for i in range(y_true.size)]))


def per_signer_accuracy(y_true, y_pred, signers) -> Dict[str, float]:
    out = {}
    for signer in sorted(set(signers.tolist()), key=lambda s: int(str(s).lstrip("p"))):
        mask = signers == signer
        out[str(signer)] = float(accuracy_score(y_true[mask], y_pred[mask]))
    return out


def metric_block(y_true, y_pred, proba, signers) -> Dict[str, Any]:
    signer_accuracy = per_signer_accuracy(y_true, y_pred, signers)
    values = np.array(list(signer_accuracy.values()))
    worst = min(signer_accuracy, key=signer_accuracy.get)
    best = max(signer_accuracy, key=signer_accuracy.get)
    return {
        "n": int(y_true.size),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted",
                                      zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        f"top{TOP_K}_accuracy": top_k_accuracy(proba, y_true),
        "signer_macro_accuracy": float(values.mean()),
        "signer_min_accuracy": float(values.min()), "signer_min_id": worst,
        "signer_max_accuracy": float(values.max()), "signer_max_id": best,
        "signer_spread": float(values.max() - values.min()),
        "signer_sd": float(values.std(ddof=1)),
        "per_signer_accuracy": signer_accuracy,
    }


# ---- pooled signer-independent metrics -------------------------------------
POOLED: Dict[Tuple[str, str], Dict[str, Any]] = {}
FOLD_ACCURACY: Dict[Tuple[str, str], np.ndarray] = {}
primary_rows, fold_rows, signer_rows = [], [], []

for model in MODELS:
    for branch in BRANCHES:
        block = OOF[(model, branch)]
        metrics = metric_block(block["y_true"], block["y_pred"], block["proba"],
                               block["signer_id"])
        POOLED[(model, branch)] = metrics

        folds = np.array([
            float(accuracy_score(block["y_true"][block["fold"] == f],
                                 block["y_pred"][block["fold"] == f]))
            for f in range(N_FOLDS)])
        FOLD_ACCURACY[(model, branch)] = folds
        primary_rows.append({
            "model": model, "representation": branch,
            "time_base": MODEL_SPEC[model]["time_base"], "n": N_CLIPS,
            "accuracy": metrics["accuracy"], "macro_f1": metrics["macro_f1"],
            "signer_macro_accuracy": metrics["signer_macro_accuracy"],
            "weighted_f1": metrics["weighted_f1"],
            "balanced_accuracy": metrics["balanced_accuracy"],
            f"top{TOP_K}_accuracy": metrics[f"top{TOP_K}_accuracy"],
            "fold_accuracy_mean": float(folds.mean()),
            "fold_accuracy_sd": float(folds.std(ddof=1)),
            "signer_min_accuracy": metrics["signer_min_accuracy"],
            "signer_min_id": metrics["signer_min_id"],
            "signer_max_accuracy": metrics["signer_max_accuracy"],
            "signer_max_id": metrics["signer_max_id"],
            "signer_spread": metrics["signer_spread"],
            "signer_sd": metrics["signer_sd"],
        })
        for fold in range(N_FOLDS):
            fold_rows.append({
                "model": model, "representation": branch, "fold": fold,
                "test_signers": " ".join(FOLDS[fold]["test"]),
                "n": int((block["fold"] == fold).sum()),
                "accuracy": float(folds[fold])})
        for signer, accuracy in metrics["per_signer_accuracy"].items():
            mask = block["signer_id"] == signer
            signer_rows.append({
                "model": model, "representation": branch, "signer_id": signer,
                "n": int(mask.sum()), "accuracy": accuracy,
                "fold_tested_in": int(block["fold"][mask][0]),
                f"top{TOP_K}_accuracy": top_k_accuracy(block["proba"][mask],
                                                       block["y_true"][mask])})

primary = pd.DataFrame(primary_rows)
fold_metrics = pd.DataFrame(fold_rows)
signer_metrics = pd.DataFrame(signer_rows)

print("=" * 78)
print("PRIMARY RESULT — SIGNER-INDEPENDENT RECOGNITION OF COMPLETELY UNSEEN SIGNERS")
print("=" * 78)
print(f"Every one of the {N_CLIPS:,} predictions in each row below was produced by a model")
print("that never trained or validated on that clip's signer. All six model x")
print("representation configurations are shown; neither representation is a")
print("pre-designated primary configuration.\n")
print(primary[["model", "representation", "accuracy", "macro_f1",
               "signer_macro_accuracy", "fold_accuracy_mean", "fold_accuracy_sd"]]
      .round(4).to_string(index=False))
print("\nSecondary metrics")
print(primary[["model", "representation", "weighted_f1", "balanced_accuracy",
               f"top{TOP_K}_accuracy"]].round(4).to_string(index=False))
print("\nSigner-generalisation, descriptive (all 16 signers held out exactly once)")
print(primary[["model", "representation", "signer_min_accuracy", "signer_min_id",
               "signer_max_accuracy", "signer_max_id", "signer_spread", "signer_sd"]]
      .round(4).to_string(index=False))

# ---- best observed configuration, identified from the recomputed values -----
best = primary.loc[primary[PRIMARY_METRIC].idxmax()]
print(f"\nBEST OBSERVED CONFIGURATION on the primary metric ({PRIMARY_METRIC}):")
print(f"  {best['model']} with the {best['representation']} representation — "
      f"{PRIMARY_METRIC} {best[PRIMARY_METRIC]:.4f}, macro-F1 {best['macro_f1']:.4f}, "
      f"signer-macro {best['signer_macro_accuracy']:.4f}")
print("  Identified from the recomputed values, not pre-designated.")
BEST_CONFIGURATION = {"model": str(best["model"]),
                      "representation": str(best["representation"]),
                      PRIMARY_METRIC: float(best[PRIMARY_METRIC]),
                      "macro_f1": float(best["macro_f1"]),
                      "signer_macro_accuracy": float(best["signer_macro_accuracy"])}

print("\nALL 16 HELD-OUT-SIGNER ACCURACIES")
signer_pivot = signer_metrics.pivot_table(
    index="signer_id", columns=["model", "representation"], values="accuracy")
signer_pivot = signer_pivot.reindex(SIGNER_ORDER)
print(signer_pivot.round(4).to_string())

print("\nPER-FOLD ACCURACY (each fold = two completely unseen signers)")
fold_pivot = fold_metrics.pivot_table(index="fold",
                                      columns=["model", "representation"],
                                      values="accuracy")
print(fold_pivot.round(4).to_string())

# ---- cross-check against the upstream per-model CSVs -----------------------
print("\nCROSS-CHECK AGAINST THE UPSTREAM PER-MODEL TABLES")
worst_pooled, worst_fold = 0.0, 0.0
for model in MODELS:
    prefix = MODEL_SPEC[model]["prefix"]
    upstream_pooled = pd.read_csv(MODEL_DIR[model] / f"{prefix}_pooled_metrics.csv")
    upstream_folds = pd.read_csv(MODEL_DIR[model] / f"{prefix}_fold_metrics.csv")
    for branch in BRANCHES:
        recorded = float(upstream_pooled.loc[upstream_pooled["branch"] == branch,
                                             "accuracy"].iloc[0])
        worst_pooled = max(worst_pooled, abs(recorded - POOLED[(model, branch)]["accuracy"]))
        recorded_folds = (upstream_folds[upstream_folds["branch"] == branch]
                          .sort_values("fold")["accuracy"].to_numpy(dtype=float))
        worst_fold = max(worst_fold,
                         float(np.abs(recorded_folds - FOLD_ACCURACY[(model, branch)]).max()))
    print(f"  {model:<7} pooled and per-fold accuracies agree with the upstream CSVs")
require("recomputed pooled accuracies agree with every upstream table",
        worst_pooled < 1e-9, f"max diff {worst_pooled:.2e}")
require("recomputed per-fold accuracies agree with every upstream table",
        worst_fold < 1e-9, f"max diff {worst_fold:.2e}")
require("no hypothesis test was invented for signer independence", True,
        "the evidence is the design plus measured unseen-signer performance")

In [ ]:
# ============================================================
# CELL 5 — Model-family comparison: DESCRIPTIVE ONLY
# ============================================================
# No cross-model significance test is run. None was pre-specified, and the families
# do not share a time base, so architecture and time base are confounded.

family_rows = []
for model in MODELS:
    for branch in BRANCHES:
        metrics = POOLED[(model, branch)]
        folds = FOLD_ACCURACY[(model, branch)]
        family_rows.append({
            "model": model, "representation": branch,
            "input_features_per_frame": 1759 if branch == "full" else 631,
            "time_base": MODEL_SPEC[model]["time_base"],
            "accuracy": metrics["accuracy"], "macro_f1": metrics["macro_f1"],
            "signer_macro_accuracy": metrics["signer_macro_accuracy"],
            f"top{TOP_K}_accuracy": metrics[f"top{TOP_K}_accuracy"],
            "fold_accuracy_mean": float(folds.mean()),
            "fold_accuracy_sd": float(folds.std(ddof=1)),
            "signer_spread": metrics["signer_spread"],
            "inferential_comparison": "NOT TESTED — descriptive only",
        })
model_family = pd.DataFrame(family_rows)

print("MODEL-FAMILY COMPARISON UNDER THE SIGNER-INDEPENDENT PROTOCOL — DESCRIPTIVE")
print(model_family[["model", "representation", "accuracy", "macro_f1",
                    "signer_macro_accuracy", f"top{TOP_K}_accuracy", "signer_spread"]]
      .round(4).to_string(index=False))

ranking = model_family.sort_values(PRIMARY_METRIC, ascending=False)
print(f"\nRanking by the primary metric ({PRIMARY_METRIC}), descriptive:")
for position, row in enumerate(ranking.itertuples(), start=1):
    print(f"  {position}. {row.model:<7} {row.representation:<8} "
          f"{getattr(row, PRIMARY_METRIC):.4f}")

by_model = (model_family.groupby("model")[PRIMARY_METRIC].max()
            .sort_values(ascending=False))
print(f"\nBest observed {PRIMARY_METRIC} per model family:")
for model, value in by_model.items():
    print(f"  {model:<7} {value:.4f}")
leader = str(by_model.index[0])
print(f"\n  {leader} achieved the highest observed signer-independent performance among")
print("  the three evaluated model families. This is a DESCRIPTIVE statement.")

print("\n  MANDATORY CAVEAT, carried into every cross-model table:")
print("  Random Forest consumed engineered per-clip summaries in PHYSICAL TIME (motion")
print("  scaled by each clip's frame rate), while BiLSTM and BiGRU consumed ordered")
print("  NATIVE-FRAME sequences. Architecture and time base are therefore confounded,")
print("  so model-family differences cannot be attributed purely to architecture.")
print("  No cross-model significance test is run: none was pre-specified, and a")
print("  significant difference would not be interpretable under this confound.")

print("\n  Signer spread is wide in every configuration "
      f"({model_family['signer_spread'].min():.4f} to "
      f"{model_family['signer_spread'].max():.4f} accuracy), so variability across")
print("  unseen signers is a first-order property of this problem rather than noise")
print("  around a single number.")

require("no cross-model significance test was computed", True,
        "descriptive comparison only, as pre-specified")
require("the model-family table records the time-base confound for every row",
        model_family["time_base"].notna().all()
        and model_family["inferential_comparison"].eq(
            "NOT TESTED — descriptive only").all(), "recorded")

In [ ]:
# ============================================================
# CELL 6 — SECONDARY: does the Reduced representation preserve or improve
#          signer-independent recognition?
# ============================================================
# Clearly secondary to the primary signer-independent question. Descriptive content
# here; the frozen inferential procedure follows in cell 7.

PAIRED_DIFFERENCES: Dict[str, np.ndarray] = {}
secondary_rows = []
for model in MODELS:
    full_folds = FOLD_ACCURACY[(model, "full")]
    reduced_folds = FOLD_ACCURACY[(model, "reduced")]
    differences = reduced_folds - full_folds            # Reduced minus Full
    PAIRED_DIFFERENCES[model] = differences
    pooled_difference = (POOLED[(model, "reduced")]["accuracy"]
                         - POOLED[(model, "full")]["accuracy"])
    secondary_rows.append({
        "model": model,
        "pooled_accuracy_full": POOLED[(model, "full")]["accuracy"],
        "pooled_accuracy_reduced": POOLED[(model, "reduced")]["accuracy"],
        "pooled_difference_reduced_minus_full": pooled_difference,
        "fold_mean_difference": float(differences.mean()),
        "fold_difference_sd": float(differences.std(ddof=1)),
        "fold_difference_min": float(differences.min()),
        "fold_difference_max": float(differences.max()),
        "folds_favouring_reduced": int((differences > 0).sum()),
        "folds_total": N_FOLDS,
        "macro_f1_full": POOLED[(model, "full")]["macro_f1"],
        "macro_f1_reduced": POOLED[(model, "reduced")]["macro_f1"],
        "macro_f1_difference": (POOLED[(model, "reduced")]["macro_f1"]
                               - POOLED[(model, "full")]["macro_f1"]),
        "signer_macro_difference": (POOLED[(model, "reduced")]["signer_macro_accuracy"]
                                    - POOLED[(model, "full")]["signer_macro_accuracy"]),
    })
secondary = pd.DataFrame(secondary_rows)

print("=" * 78)
print("SECONDARY ANALYSIS — FULL vs REDUCED FACE REPRESENTATION")
print("=" * 78)
print("Secondary to the primary signer-independent question. Direction throughout is")
print("REDUCED minus FULL, derived here from the saved predictions.\n")
print(secondary[["model", "pooled_accuracy_full", "pooled_accuracy_reduced",
                 "pooled_difference_reduced_minus_full", "fold_mean_difference",
                 "folds_favouring_reduced", "folds_total"]].round(4).to_string(index=False))

print("\nThe fold mean is UNWEIGHTED across the 8 folds, which have unequal test sizes,")
print("so it differs slightly from the pooled difference. The fold mean is the")
print("inferential statistic; the pooled difference is descriptive.")

print("\nPER-FOLD PAIRED DIFFERENCES (Reduced minus Full accuracy)")
difference_table = pd.DataFrame(PAIRED_DIFFERENCES, index=[f"fold {k}" for k in range(N_FOLDS)])
print(difference_table.round(4).to_string())
print(difference_table.agg(["mean", "std", "min", "max"]).round(4).to_string())

print("\nCLASS-BALANCED AND SIGNER-GENERALISATION DIFFERENCES (reported, not tested)")
print(secondary[["model", "macro_f1_full", "macro_f1_reduced", "macro_f1_difference",
                 "signer_macro_difference"]].round(4).to_string(index=False))
print("  Macro-F1 receives NO significance-test family. Accuracy is the only")
print("  inferential metric for this secondary hypothesis.")

# ---- exploratory signer-level differences ----------------------------------
print("\nSIGNER-LEVEL DIFFERENCES — EXPLORATORY / DESCRIPTIVE ONLY")
signer_difference_rows = []
for model in MODELS:
    full_map = POOLED[(model, "full")]["per_signer_accuracy"]
    reduced_map = POOLED[(model, "reduced")]["per_signer_accuracy"]
    for signer in SIGNER_ORDER:
        signer_difference_rows.append({
            "model": model, "signer_id": signer,
            "accuracy_full": full_map[signer], "accuracy_reduced": reduced_map[signer],
            "difference_reduced_minus_full": reduced_map[signer] - full_map[signer],
            "status": "EXPLORATORY"})
signer_differences = pd.DataFrame(signer_difference_rows)
exploratory = signer_differences.pivot(index="signer_id", columns="model",
                                       values="difference_reduced_minus_full")
print(exploratory.reindex(SIGNER_ORDER).round(4).to_string())
for model in MODELS:
    subset = signer_differences[signer_differences["model"] == model]
    positive = int((subset["difference_reduced_minus_full"] > 0).sum())
    print(f"  {model:<7} signers favouring reduced: {positive} of {N_SIGNERS} | "
          f"range {subset['difference_reduced_minus_full'].min():+.4f} to "
          f"{subset['difference_reduced_minus_full'].max():+.4f}")
print("\n  EXPLORATORY, not inferential: the two signers in a fold share one trained")
print("  model and folds share training signers, so the 16 observations per model are")
print("  not independent. No test is applied to them.")

require("paired differences were derived as Reduced minus Full from the predictions",
        all(np.allclose(PAIRED_DIFFERENCES[m],
                        FOLD_ACCURACY[(m, "reduced")] - FOLD_ACCURACY[(m, "full")])
            for m in MODELS), "sign convention derived, not inherited")
require("signer-level differences are labelled exploratory",
        set(signer_differences["status"].unique()) == {"EXPLORATORY"}, "EXPLORATORY")

In [ ]:
# ============================================================
# CELL 7 — SECONDARY inference: exhaustive paired sign-flip, Holm, bootstrap
# ============================================================
# The pre-registered procedure, applied to the SECONDARY representation question.
# Accuracy only. Nothing hard-coded: every quantity comes from cell 6's recomputed
# fold differences.

SIGN_ASSIGNMENTS = np.array(list(itertools.product([1.0, -1.0], repeat=N_FOLDS)))
require("all sign assignments were enumerated exhaustively",
        SIGN_ASSIGNMENTS.shape == (SIGN_FLIP_ASSIGNMENTS, N_FOLDS),
        f"{SIGN_ASSIGNMENTS.shape[0]} = 2^{N_FOLDS}")


def sign_flip_test(differences: np.ndarray) -> Dict[str, Any]:
    """Exhaustive two-sided paired sign-flip permutation test."""
    observed = float(differences.mean())
    means = SIGN_ASSIGNMENTS @ differences / differences.size
    extreme = int(np.sum(np.abs(means) >= abs(observed) - 1e-12))
    return {"statistic_T": observed, "assignments": int(means.size),
            "n_at_least_as_extreme": extreme, "p_value": extreme / means.size}


def holm_adjust(p_values: Sequence[float]) -> np.ndarray:
    """Step-down Holm with monotonicity enforced. Valid under arbitrary dependence."""
    values = np.asarray(p_values, dtype=float)
    m = values.size
    order = np.argsort(values, kind="mergesort")
    adjusted = np.empty(m)
    running = 0.0
    for position, index in enumerate(order):
        running = max(running, min(1.0, (m - position) * values[index]))
        adjusted[index] = running
    return adjusted


def fold_clustered_bootstrap(differences: np.ndarray) -> Dict[str, float]:
    """Resample the 8 folds as clusters. Fresh generator per model on the frozen seed,
    so all three models see the same cluster resamples and remain comparable."""
    generator = np.random.default_rng(BOOTSTRAP_SEED)
    indices = generator.integers(0, differences.size,
                                 size=(BOOTSTRAP_RESAMPLES, differences.size))
    means = differences[indices].mean(axis=1)
    lower = (1.0 - BOOTSTRAP_CI) / 2.0
    low, high = np.quantile(means, [lower, 1.0 - lower])
    return {"ci_low": float(low), "ci_high": float(high),
            "bootstrap_mean": float(means.mean()),
            "bootstrap_sd": float(means.std(ddof=1)),
            "share_above_zero": float(np.mean(means > 0.0))}


print("=" * 78)
print("SECONDARY INFERENCE — FULL vs REDUCED, ACCURACY ONLY")
print("=" * 78)
print(f"Statistic  : mean of the {N_FOLDS} paired Reduced-minus-Full fold accuracy "
      "differences")
print(f"Test       : exhaustive two-sided paired sign-flip, {SIGN_FLIP_ASSIGNMENTS} "
      f"assignments")
print(f"Minimum p  : {SIGN_FLIP_MIN_P:.7f} (a property of n = {N_FOLDS})")
print(f"Sensitivity: {BOOTSTRAP_RESAMPLES:,} resamples, seed {BOOTSTRAP_SEED}, "
      f"{BOOTSTRAP_UNIT}\n")

inference_rows = []
for model in MODELS:
    differences = PAIRED_DIFFERENCES[model]
    test = sign_flip_test(differences)
    bootstrap = fold_clustered_bootstrap(differences)
    inference_rows.append({
        "model": model, "metric": INFERENCE_METRIC,
        "statistic_T_reduced_minus_full": test["statistic_T"],
        "fold_difference_sd": float(differences.std(ddof=1)),
        "fold_difference_min": float(differences.min()),
        "fold_difference_max": float(differences.max()),
        "folds_favouring_reduced": int((differences > 0).sum()),
        "assignments_enumerated": test["assignments"],
        "n_at_least_as_extreme": test["n_at_least_as_extreme"],
        "p_raw": test["p_value"],
        "bootstrap_ci_low": bootstrap["ci_low"],
        "bootstrap_ci_high": bootstrap["ci_high"],
        "bootstrap_sd": bootstrap["bootstrap_sd"],
        "bootstrap_share_above_zero": bootstrap["share_above_zero"],
    })
inference = pd.DataFrame(inference_rows)
inference["p_holm"] = holm_adjust(inference["p_raw"].to_numpy())
inference["multiplicity"] = MULTIPLICITY
inference["equivalence_claim"] = "NOT MADE — no margin was pre-specified"

print("EFFECT SIZES FIRST (the reporting rule: p cannot distinguish small from large "
      "at n = 8)")
print(inference[["model", "statistic_T_reduced_minus_full", "fold_difference_sd",
                 "fold_difference_min", "fold_difference_max",
                 "folds_favouring_reduced", "bootstrap_ci_low", "bootstrap_ci_high"]]
      .round(5).to_string(index=False))

print("\nPERMUTATION TEST, RAW AND HOLM-ADJUSTED")
print(inference[["model", "assignments_enumerated", "n_at_least_as_extreme",
                 "p_raw", "p_holm"]].to_string(index=False))

print(f"\n  Pooled differences for comparison (descriptive, size-weighted):")
for model in MODELS:
    pooled_difference = (POOLED[(model, "reduced")]["accuracy"]
                         - POOLED[(model, "full")]["accuracy"])
    print(f"    {model:<7} fold-mean T {PAIRED_DIFFERENCES[model].mean():+.5f} | "
          f"pooled {pooled_difference:+.5f}")

print("\n  INTERPRETATION CLAUSES, printed with every result:")
print("  - The 8 folds are NOT independent: consecutive folds share 10 of 12 training")
print("    signers, so the enumeration is exhaustive but not 'exact' under independence.")
print(f"  - Power is low with n = {N_FOLDS}. The permutation p has a hard floor of")
print(f"    {SIGN_FLIP_MIN_P:.7f} and cannot distinguish a small effect from a large one,")
print("    which is why the effect sizes and interval above lead the reporting.")
print("  - Holm controls the family-wise error rate under ARBITRARY DEPENDENCE and does")
print("    NOT require independence. The three tests are related replications of one")
print("    hypothesis in three model families, not three independent attempts.")
print("  - The fold-clustered bootstrap has only 8 clusters, so the interval is coarse")
print("    and unstable. It is a robustness check on the sign of the effect, not a")
print("    precise interval.")
print("  - A small p would not identify the MECHANISM behind any difference.")
print("  - NON-SIGNIFICANCE WOULD NOT DEMONSTRATE EQUIVALENCE. No equivalence margin")
print("    was ever pre-specified, so no equivalence claim is made.")

require("the sign-flip test enumerated every assignment for all three models",
        (inference["assignments_enumerated"] == SIGN_FLIP_ASSIGNMENTS).all(),
        f"{SIGN_FLIP_ASSIGNMENTS} each")
require("every raw p is at or above the attainable minimum",
        (inference["p_raw"] >= SIGN_FLIP_MIN_P - 1e-12).all(),
        f"floor {SIGN_FLIP_MIN_P:.7f}")
require("Holm-adjusted p values are never below the raw values",
        (inference["p_holm"] >= inference["p_raw"] - 1e-12).all(), "monotone")
require("Holm-adjusted p values are non-decreasing in the raw ordering",
        np.all(np.diff(inference.sort_values("p_raw")["p_holm"].to_numpy()) >= -1e-12),
        "step-down enforced")
require("accuracy is the only metric given an inferential test",
        INFERENCE_METRIC == "accuracy"
        and set(inference["metric"].unique()) == {"accuracy"}, "macro-F1 untested")
require("no equivalence claim is recorded",
        inference["equivalence_claim"].str.startswith("NOT MADE").all(), "forbidden")

In [ ]:
# ============================================================
# CELL 8 — SECONDARY / DIAGNOSTIC: the signer-overlap performance gap
# ============================================================
# Supports the primary signer-independent thesis by quantifying how much a
# non-signer-independent evaluation would have overstated performance.

control_rows = []
for model in MODELS:
    for branch in BRANCHES:
        block = CONTROL[(model, branch)]
        overlap_accuracy = float(accuracy_score(block["y_true"], block["y_pred"]))
        independent_accuracy = POOLED[(model, branch)]["accuracy"]
        control_rows.append({
            "model": model, "representation": branch,
            "kind": "INTENTIONAL SIGNER-OVERLAP CONTROL",
            "valid_as_signer_independent": False,
            "n_control_test": int(block["y_true"].size),
            "signer_overlap_accuracy": overlap_accuracy,
            "signer_independent_oof_accuracy": independent_accuracy,
            "signer_overlap_performance_gap": overlap_accuracy - independent_accuracy,
            "control_macro_f1": float(f1_score(block["y_true"], block["y_pred"],
                                               average="macro", zero_division=0)),
        })
control_gap = pd.DataFrame(control_rows)

print("=" * 78)
print("SIGNER-OVERLAP PERFORMANCE GAP — SECONDARY / DIAGNOSTIC")
print("=" * 78)
print("The overlap is INTENTIONAL and by design. It is not leakage: it is a deliberately")
print("signer-dependent control whose only purpose is to quantify what a")
print("non-signer-independent evaluation would have reported.\n")
print(control_gap[["model", "representation", "signer_overlap_accuracy",
                   "signer_independent_oof_accuracy",
                   "signer_overlap_performance_gap"]].round(4).to_string(index=False))

neural_gaps = control_gap.loc[control_gap["model"] != "RF",
                              "signer_overlap_performance_gap"]
rf_gaps = control_gap.loc[control_gap["model"] == "RF",
                          "signer_overlap_performance_gap"]
print(f"\n  Random Forest gaps : {rf_gaps.min():+.4f} to {rf_gaps.max():+.4f}")
print(f"  neural model gaps  : {neural_gaps.min():+.4f} to {neural_gaps.max():+.4f}")
print("\n  The signer-overlap performance gap was substantially larger for the neural")
print("  sequence models than for Random Forest.")

print("\n  WHAT THE GAP DOES AND DOES NOT SHOW")
print("  It shows that when the same signers appear in training and testing, measured")
print("  accuracy rises. It therefore demonstrates why evaluating on genuinely unseen")
print("  signers is necessary: a signer-dependent protocol would have reported")
print("  materially higher numbers for the same models and data.")
print("\n  The gap is NOT attributed wholly to signer identity memorisation. Plausible")
print("  and unseparated contributors include:")
for contributor in ("signer identity", "signing style", "session conditions",
                    "background similarity", "other signer-associated cues"):
    print(f"    - {contributor}")
print("  This analysis does not separate them, and no causal attribution is made.")
print("\n  It also does NOT show that the representation is free of signer information —")
print("  quite the opposite. Signer-associated cues are present and usable, which is")
print("  precisely the reason the signer-independent protocol is required.")

require("control results are never pooled with the signer-independent out-of-fold sets",
        all(CONTROL[(m, b)]["y_true"].size == CONTROL_SPLIT_SIZES["test"]
            for m in MODELS for b in BRANCHES)
        and all(OOF[(m, b)]["clip_id"].size == N_CLIPS
                for m in MODELS for b in BRANCHES),
        "750 control test clips vs 6,000 out-of-fold predictions, separate arrays")
require("every control row is flagged invalid as a signer-independent evaluation",
        (~control_gap["valid_as_signer_independent"]).all(), "by design")
require("the gap is reported as a performance gap, never as leakage", True,
        "intentional overlap, contributors unseparated")

In [ ]:
# ============================================================
# CELL 9 — Efficiency: a formal SECONDARY research outcome
# ============================================================
# Reported separately per model family. No universal computational advantage is
# claimed, because the measured saving differs sharply between families.

FOLD_SCOPES = {str(k) for k in range(N_FOLDS)}
efficiency_rows = []

for model in MODELS:
    prefix = MODEL_SPEC[model]["prefix"]
    table = pd.read_csv(MODEL_DIR[model] / f"{prefix}_efficiency.csv")
    table["scope"] = table["scope"].astype(str)
    primary_scopes = table[table["scope"].isin(FOLD_SCOPES)]
    require(f"{model} efficiency table covers 8 folds per branch",
            len(primary_scopes) == N_FOLDS * len(BRANCHES), f"{len(primary_scopes)} rows")

    time_column = "fit_seconds" if model == "RF" else "train_seconds"
    row: Dict[str, Any] = {"model": model,
                           "training_time_column": time_column,
                           "input_features_per_frame_full": 1759,
                           "input_features_per_frame_reduced": 631,
                           "input_feature_ratio": round(1759 / 631, 4)}
    for branch in BRANCHES:
        branch_scopes = primary_scopes[primary_scopes["branch"] == branch]
        row[f"mean_training_seconds_{branch}"] = float(branch_scopes[time_column].mean())
        if model == "RF":
            row[f"model_input_dimension_{branch}"] = int(branch_scopes["features"].iloc[0])
            sizes = branch_scopes["serialized_model_mib"].dropna()
            row[f"serialized_model_mib_{branch}"] = (float(sizes.iloc[0])
                                                    if len(sizes) else None)
        else:
            row[f"parameters_{branch}"] = int(branch_scopes["parameters"].iloc[0])
            row[f"ram_cache_gib_{branch}"] = float(branch_scopes["cache_gib"].mean())
            row[f"peak_gpu_gib_{branch}"] = float(branch_scopes["peak_gpu_gib"].max())
            row[f"mean_epochs_{branch}"] = float(branch_scopes["epochs_run"].mean())
    row["training_time_ratio_full_over_reduced"] = (
        row["mean_training_seconds_full"] / row["mean_training_seconds_reduced"])
    if model != "RF":
        row["parameter_difference"] = row["parameters_full"] - row["parameters_reduced"]
        row["ram_cache_ratio_full_over_reduced"] = (
            row["ram_cache_gib_full"] / row["ram_cache_gib_reduced"])
    else:
        row["model_input_dimension_ratio"] = (row["model_input_dimension_full"]
                                              / row["model_input_dimension_reduced"])
    efficiency_rows.append(row)

efficiency = pd.DataFrame(efficiency_rows)

print("=" * 78)
print("EFFICIENCY — FORMAL SECONDARY RESEARCH OUTCOME")
print("=" * 78)
print("Input representation is 1,759 features per frame (Full) versus 631 (Reduced), a")
print(f"ratio of {1759 / 631:.3f}x. What that buys differs sharply by model family.\n")
print("Training-time saving from the Reduced representation, per family")
print(efficiency[["model", "mean_training_seconds_full", "mean_training_seconds_reduced",
                  "training_time_ratio_full_over_reduced"]].round(3).to_string(index=False))

print("\nRandom Forest specifics")
rf_row = efficiency[efficiency["model"] == "RF"].iloc[0]
print(f"  model-input dimension : {int(rf_row['model_input_dimension_full']):,} -> "
      f"{int(rf_row['model_input_dimension_reduced']):,} "
      f"({rf_row['model_input_dimension_ratio']:.3f}x)")
print(f"  mean fit time         : {rf_row['mean_training_seconds_full']:.0f}s -> "
      f"{rf_row['mean_training_seconds_reduced']:.0f}s "
      f"({rf_row['training_time_ratio_full_over_reduced']:.3f}x)")
if rf_row.get("serialized_model_mib_full") is not None:
    print(f"  representative model  : {rf_row['serialized_model_mib_full']:,.0f} MiB -> "
          f"{rf_row['serialized_model_mib_reduced']:,.0f} MiB")

print("\nNeural specifics")
for model in ("BiLSTM", "BiGRU"):
    row = efficiency[efficiency["model"] == model].iloc[0]
    print(f"  {model}")
    print(f"    parameters   : {int(row['parameters_full']):,} -> "
          f"{int(row['parameters_reduced']):,} "
          f"(difference {int(row['parameter_difference']):,}, all in the input "
          "projection)")
    print(f"    mean training: {row['mean_training_seconds_full'] / 60:.1f} min -> "
          f"{row['mean_training_seconds_reduced'] / 60:.1f} min "
          f"({row['training_time_ratio_full_over_reduced']:.3f}x)")
    print(f"    RAM cache    : {row['ram_cache_gib_full']:.2f} -> "
          f"{row['ram_cache_gib_reduced']:.2f} GiB "
          f"({row['ram_cache_ratio_full_over_reduced']:.2f}x)")
    print(f"    peak GPU     : {row['peak_gpu_gib_full']:.2f} -> "
          f"{row['peak_gpu_gib_reduced']:.2f} GiB")
    print(f"    mean epochs  : {row['mean_epochs_full']:.1f} -> "
          f"{row['mean_epochs_reduced']:.1f}")

fastest = efficiency.loc[efficiency["training_time_ratio_full_over_reduced"].idxmax()]
slowest = efficiency.loc[efficiency["training_time_ratio_full_over_reduced"].idxmin()]
print(f"\n  NO UNIVERSAL COMPUTATIONAL ADVANTAGE IS CLAIMED.")
print(f"  The measured training-time saving ranges from "
      f"{slowest['training_time_ratio_full_over_reduced']:.2f}x ({slowest['model']}) to "
      f"{fastest['training_time_ratio_full_over_reduced']:.2f}x ({fastest['model']}).")
print("  For Random Forest the saving is small because max_features was fixed at 96, so")
print("  per-split work is nearly independent of the total feature count. The neural")
print("  savings are real but confined to the input projection and the data pipeline;")
print("  the recurrent core is identical in both branches.")
print("  Each figure must be quoted per model family, never generalised.")

require("efficiency is reported separately for every model family",
        len(efficiency) == len(MODELS), f"{len(efficiency)} families")
require("no universal computational saving is asserted",
        efficiency["training_time_ratio_full_over_reduced"].nunique() > 1,
        "family-specific ratios reported")

In [ ]:
# ============================================================
# CELL 10 — Per-class results: concise summary in main output, detail supplementary
# ============================================================

class_rows, summary_rows = [], []
CONFUSION: Dict[Tuple[str, str], np.ndarray] = {}
for model in MODELS:
    prefix = MODEL_SPEC[model]["prefix"]
    for branch in BRANCHES:
        block = OOF[(model, branch)]
        recalls = recall_score(block["y_true"], block["y_pred"], average=None,
                              labels=np.arange(N_LABELS), zero_division=0)
        support = np.array([int((block["y_true"] == c).sum()) for c in range(N_LABELS)])
        for label_id in range(N_LABELS):
            class_rows.append({
                "model": model, "representation": branch, "label_id": label_id,
                "label": ID_TO_LABEL[label_id], "recall": float(recalls[label_id]),
                "support": int(support[label_id]), "status": "SUPPLEMENTARY"})

        matrix = confusion_matrix(block["y_true"], block["y_pred"],
                                 labels=np.arange(N_LABELS)).astype(np.int32)
        CONFUSION[(model, branch)] = matrix
        np.save(WORKING / f"final_confusion_{prefix}_{branch}.npy", matrix)
        require(f"{model} {branch} confusion matrix is 200x200 and totals 6,000",
                matrix.shape == (N_LABELS, N_LABELS) and int(matrix.sum()) == N_CLIPS,
                f"{matrix.shape}, {int(matrix.sum()):,}")
        require(f"{model} {branch} pooled accuracy reproduces from the confusion matrix",
                abs(float(np.trace(matrix) / N_CLIPS)
                    - POOLED[(model, branch)]["accuracy"]) < 1e-12,
                f"{POOLED[(model, branch)]['accuracy']:.6f}")

        order = np.argsort(recalls, kind="mergesort")
        summary_rows.append({
            "model": model, "representation": branch,
            "mean_class_recall": float(recalls.mean()),
            "median_class_recall": float(np.median(recalls)),
            "classes_at_zero_recall": int((recalls == 0.0).sum()),
            f"classes_below_{int(CLASS_RECALL_THRESHOLD * 100)}pct_recall":
                int((recalls < CLASS_RECALL_THRESHOLD).sum()),
            "classes_at_perfect_recall": int((recalls == 1.0).sum()),
            "hardest_classes": ", ".join(
                f"{ID_TO_LABEL[int(c)]}({recalls[c]:.2f})"
                for c in order[:CLASS_SUMMARY_TOP_N]),
            "easiest_classes": ", ".join(
                f"{ID_TO_LABEL[int(c)]}({recalls[c]:.2f})"
                for c in order[::-1][:CLASS_SUMMARY_TOP_N]),
        })

class_recall = pd.DataFrame(class_rows)
class_summary = pd.DataFrame(summary_rows)

print("PER-CLASS SUMMARY (main output — full tables are supplementary)")
print(class_summary[["model", "representation", "mean_class_recall",
                     "median_class_recall", "classes_at_zero_recall",
                     f"classes_below_{int(CLASS_RECALL_THRESHOLD * 100)}pct_recall",
                     "classes_at_perfect_recall"]].round(4).to_string(index=False))

print(f"\nHardest {CLASS_SUMMARY_TOP_N} classes by recall, best observed configuration "
      f"({BEST_CONFIGURATION['model']} {BEST_CONFIGURATION['representation']})")
best_summary = class_summary[
    (class_summary["model"] == BEST_CONFIGURATION["model"])
    & (class_summary["representation"] == BEST_CONFIGURATION["representation"])].iloc[0]
print(f"  {best_summary['hardest_classes']}")
print(f"Easiest {CLASS_SUMMARY_TOP_N}")
print(f"  {best_summary['easiest_classes']}")

print(f"\n  Full 200-row per-class tables and all six 200x200 confusion matrices are")
print("  written as SUPPLEMENTARY artefacts, not reproduced in the main results.")
require("per-class detail is labelled supplementary",
        set(class_recall["status"].unique()) == {"SUPPLEMENTARY"}, "SUPPLEMENTARY")
require("per-class tables cover every class for all six configurations",
        len(class_recall) == N_LABELS * len(MODELS) * len(BRANCHES),
        f"{len(class_recall):,} rows")

# ---- cross-check the recomputed confusion matrices against the upstream files
worst_confusion = 0
for model in MODELS:
    prefix = MODEL_SPEC[model]["prefix"]
    for branch in BRANCHES:
        upstream_path = MODEL_DIR[model] / f"{prefix}_confusion_matrix_{branch}.npy"
        if upstream_path.exists():
            upstream = np.load(upstream_path)
            worst_confusion = max(worst_confusion,
                                  int(np.abs(upstream.astype(np.int64)
                                             - CONFUSION[(model, branch)].astype(np.int64)).max()))
require("recomputed confusion matrices match the upstream ones exactly",
        worst_confusion == 0, f"max cell difference {worst_confusion}")

In [ ]:
# ============================================================
# CELL 11 — Limitations, thesis tables, frozen contract, report last
# ============================================================

LIMITATIONS = {
    "signer_population": "Only 16 signers. Every conclusion rests on 16 people, so "
                         "generalisation to the wider signing population is not "
                         "established.",
    "fold_resolution": f"{N_FOLDS} folds with 2 test signers each. The secondary "
                       f"representation test rests on {N_FOLDS} paired numbers, so the "
                       f"permutation p has a hard floor of {SIGN_FLIP_MIN_P:.7f} and very "
                       "limited resolution.",
    "overlapping_training_sets": "Consecutive folds share 10 of 12 training signers, so "
                                 "the 8 fold estimates are correlated. The enumeration is "
                                 "exhaustive but not 'exact' in the strict independence "
                                 "sense.",
    "validation_test_rotation": "Each fold's 2 validation signers are the next fold's 2 "
                                "test signers. This makes the fold estimates correlated, "
                                "but it is NOT leakage into a signer's own test model: a "
                                "signer's predictions always come from a model that "
                                "never trained or validated on them.",
    "neural_time_base": "BiLSTM and BiGRU used NATIVE FRAMES. The ~30 and ~60 fps "
                        "recording regimes track the signer, so sequence length is a "
                        "signer-correlated nuisance. Every training fold contained both "
                        "regimes, which is why the policy was accepted, but it is not "
                        "removed.",
    "rf_vs_neural_time": "Random Forest used engineered motion summaries expressed in "
                         "physical time; the neural models used native-frame sequences. "
                         "Model-family differences cannot be attributed purely to "
                         "architecture.",
    "neural_parameter_difference": "Within each neural family the Full and Reduced models "
                                   "share a parameter-identical recurrent core and head, "
                                   "which additionally initialise bit-identically, but "
                                   "total parameter counts differ by 288,768 "
                                   "input-projection weights.",
    "rf_dimensionality": "The Random Forest Full and Reduced branches differ by roughly "
                         "2.81x in model-input dimension after zero-variance filtering, "
                         "a representation-level confound.",
    "rf_max_features": "The primary Random Forest used max_features=96, approximately "
                       "0.61x Full's natural sqrt search breadth and 1.02x Reduced's, so "
                       "it is potentially conservative toward Full in per-split search "
                       "breadth. The sqrt sensitivity is pre-declared and not run here.",
    "neural_epoch_ceiling": "Some neural scopes exhausted the 60-epoch budget: in BiLSTM "
                            "two primary scopes, both Reduced; in BiGRU three primary "
                            "scopes, all Full, plus the Reduced control. Different "
                            "branches were affected in different models, so no consistent "
                            "bias direction can be attributed to the ceiling, and the "
                            "direction of its effect on unseen-signer test performance "
                            "cannot be known.",
    "cudnn_determinism": "cuDNN recurrent kernels have no deterministic implementation, "
                         "so neural prediction digests may vary between reruns. The "
                         "digests verified here are those of the frozen runs.",
    "significance_not_mechanism": "A small p-value indicates the observed difference is "
                                  "unlikely under the null. It does not identify why the "
                                  "difference occurs.",
    "no_equivalence": "Non-significance would not demonstrate equivalence. No equivalence "
                      "margin was pre-specified, so no equivalence claim is made.",
    "signer_independence_scope": "Signer-independent PERFORMANCE does not prove "
                                 "signer-invariant REPRESENTATIONS. The overlap control "
                                 "shows signer-associated cues are present and usable.",
    "generalisation": "Results cannot automatically be generalised beyond this dataset "
                      "and this signer population.",
}

DEFERRED = {
    "rf_max_features_sqrt": "pre-declared SECONDARY sensitivity; not run in notebook 13; "
                            "must never overwrite the primary Random Forest result",
    "neural_common_time_fps": "pre-declared SECONDARY sensitivity; exact method including "
                              "the target rate is NOT yet frozen and must be frozen "
                              "before execution; must never replace the primary "
                              "native-frame result",
}

FINAL_CONTRACT = {
    "notebook": "13-final-comparison-and-inference",
    "analysis_policy_version": ANALYSIS_POLICY_VERSION,
    "thesis_hierarchy": {
        "primary": "signer-independent recognition of Bangla signs produced by "
                   "completely unseen signers, across Random Forest, BiLSTM and BiGRU, "
                   f"under the frozen {N_FOLDS}-fold protocol",
        "secondary_representation": "does the Reduced-face representation (92 landmarks) "
                                    "preserve or improve signer-independent recognition "
                                    "versus Full-face (468), and what computational "
                                    "savings does it provide",
        "secondary_diagnostic": "what happens when signer overlap between training and "
                                "testing is intentionally allowed",
        "supplementary": "per-class results, confusion matrices, detailed efficiency "
                         "tables, deferred sensitivities",
    },
    "metric_hierarchy": {
        "primary_headline": f"pooled signer-independent {PRIMARY_METRIC}",
        "key_secondary": list(KEY_SECONDARY_METRICS),
        "prominent_descriptive": ["signer_min_accuracy", "signer_max_accuracy",
                                  "signer_spread", "all 16 signer accuracies"],
        "secondary": list(SECONDARY_METRICS),
        "supplementary": ["per-class recall", "200x200 confusion matrices"],
        "note": "minimum-signer accuracy is reported prominently as a descriptive "
                "signer-generalisation outcome and is NOT promoted to a primary outcome",
    },
    "configurations_reported": len(MODELS) * len(BRANCHES),
    "pre_designated_representation": None,
    "best_observed_policy": "the best observed configuration is identified from the "
                            "recomputed values and reported in the manifest and report; "
                            "it is deliberately NOT part of this contract, because the "
                            "contract records policy and must hash identically "
                            "regardless of the results",
    "upstream": {
        "experiment_id": REQUIRED_EXPERIMENT_ID,
        "loader_contract_id": REQUIRED_LOADER_CONTRACT_ID,
        "standardiser_set_id": REQUIRED_STANDARDISER_SET_ID,
        "time_metadata_id": REQUIRED_TIME_METADATA_ID,
        "feature_contract_id": REQUIRED_FEATURE_CONTRACT_ID,
        "filter_set_id": REQUIRED_FILTER_SET_ID,
        "model_contracts": {m: MODEL_SPEC[m]["contract"] for m in MODELS},
        "model_contract_verification": "each upstream model contract id was "
                                       "INDEPENDENTLY RECOMPUTED from its saved contract "
                                       "object using the upstream hashing recipe "
                                       "sha256(json.dumps(contract, sort_keys=True, "
                                       "separators=(',',':'), default=str))[:16], and "
                                       "cross-checked against the frozen expected id and "
                                       "a clean upstream validation report",
        "prediction_digests_verified": 12,
        "notebook7_hashes": dict(REQUIRED_NB7_HASHES),
    },
    "signer_independence": {
        "verified_properties": [
            f"{N_SIGNERS} signers, {N_FOLDS} folds",
            f"{TRAIN_SIGNERS_PER_FOLD} train / 2 validation / 2 test signers per fold",
            "train, validation and test signer sets disjoint in every fold",
            "every signer a test signer exactly once",
            "every signer a validation signer exactly once",
            f"the {N_FOLDS} test folds tile all {N_CLIPS} clips exactly once",
            "every out-of-fold prediction came from a fold that held out that clip's "
            "signer, and from a model that neither trained nor validated on that signer",
            "fold standardisation and the RF zero-variance filter were fitted on "
            "training signers only",
            "control predictions are stored separately and never pooled",
        ],
        "claim_made": "signer-independent EVALUATION: the protocol guarantees the model "
                      "never saw the test signer",
        "claim_not_made": "that the representation contains no signer information. The "
                          "overlap control shows signer-associated cues are present and "
                          "usable.",
    },
    "statistical_plan_secondary": {
        "statistic": f"mean of the {N_FOLDS} paired Reduced-minus-Full fold "
                     f"{INFERENCE_METRIC} differences",
        "test": "exhaustive two-sided paired sign-flip permutation over all "
                f"{SIGN_FLIP_ASSIGNMENTS} sign assignments",
        "p_definition": "proportion of assignments whose absolute mean difference is at "
                        "least the observed absolute mean difference",
        "minimum_attainable_p": SIGN_FLIP_MIN_P,
        "inferential_metric": INFERENCE_METRIC,
        "macro_f1": "reported prominently, given NO significance-test family",
        "multiplicity": MULTIPLICITY,
        "holm_note": "Holm controls the family-wise error rate under arbitrary "
                     "dependence and does not require independence. The three tests are "
                     "related replications of one hypothesis in three model families.",
        "sensitivity": {"kind": "fold-clustered bootstrap",
                        "resamples": BOOTSTRAP_RESAMPLES, "seed": BOOTSTRAP_SEED,
                        "unit": BOOTSTRAP_UNIT, "interval": f"{BOOTSTRAP_CI:.0%} percentile",
                        "status": "secondary; never replaces the permutation test",
                        "caveat": f"only {N_FOLDS} clusters, so the interval is coarse "
                                  "and unstable"},
        "clip_level_bootstrap": "not used as an inferential result",
        "equivalence": "no equivalence claim; no margin was pre-specified",
        "reporting_rule": "effect sizes, per-fold ranges and the bootstrap interval lead; "
                          "p-values are reported alongside, not as the headline",
        "nothing_hard_coded": True,
    },
    "cross_model": "DESCRIPTIVE ONLY. No cross-model significance test was pre-specified "
                   "or run. Random Forest used physical-time engineered summaries while "
                   "the neural models used native-frame sequences, so architecture and "
                   "time base are confounded.",
    "control_policy": {
        "name": "signer-overlap performance gap",
        "never_called": "leakage — the overlap is intentional and by design",
        "attribution": "not attributed wholly to signer identity memorisation; "
                       "contributors include signer identity, signing style, session "
                       "conditions, background similarity and other signer-associated "
                       "cues, and this analysis does not separate them",
        "status": "secondary / diagnostic, supporting the primary signer-independent "
                  "thesis",
    },
    "efficiency_policy": "formal secondary research outcome, reported separately per "
                         "model family; no universal computational advantage is claimed",
    "limitations": LIMITATIONS,
    "deferred_sensitivities": DEFERRED,
    "forbidden": ["retraining any model", "creating or altering folds",
                  "any hyperparameter", "cross-model significance testing",
                  "any equivalence claim",
                  "clip-level bootstrap presented as inferential",
                  "running either deferred sensitivity",
                  "hard-coding any expected p-value or fold accuracy",
                  "describing either representation as a pre-designated primary "
                  "configuration"],
}
FINAL_ANALYSIS_CONTRACT_ID = sha16(json.dumps(FINAL_CONTRACT, sort_keys=True,
                                              separators=(",", ":"), default=str))

blob = json.dumps(FINAL_CONTRACT, default=str)
require("the analysis contract carries no session path", "/kaggle" not in blob, "stable")
require("the analysis contract carries no timestamp", "generated_at" not in blob, "stable")
require("the contract records the primary thesis question as signer independence",
        FINAL_CONTRACT["thesis_hierarchy"]["primary"].startswith(
            "signer-independent recognition"), "primary")
require("the contract records the representation question as secondary",
        "secondary_representation" in FINAL_CONTRACT["thesis_hierarchy"], "secondary")
require("no representation is pre-designated",
        FINAL_CONTRACT["pre_designated_representation"] is None, "none")
require("the analysis contract holds policy only, no measured result",
        "best_observed_configuration" not in json.dumps(FINAL_CONTRACT)
        and not any(isinstance(v, float) for v in FINAL_CONTRACT.values()),
        "hashes identically regardless of the results")
require("the contract distinguishes signer-independent evaluation from signer-invariant "
        "representations",
        "does not prove signer-invariant"
        in FINAL_CONTRACT["limitations"]["signer_independence_scope"], "recorded")
require("the contract records that macro-F1 gets no test family",
        "NO significance-test family"
        in FINAL_CONTRACT["statistical_plan_secondary"]["macro_f1"], "recorded")
require("the contract records that Holm needs no independence assumption",
        "does not require independence"
        in FINAL_CONTRACT["statistical_plan_secondary"]["holm_note"], "recorded")
require("the contract forbids any equivalence claim",
        "any equivalence claim" in FINAL_CONTRACT["forbidden"], "forbidden")
require("both deferred sensitivities are recorded as not run",
        len(FINAL_CONTRACT["deferred_sensitivities"]) == 2
        and all("not" in v.lower() for v in DEFERRED.values()), "deferred")

# ---- write everything ------------------------------------------------------
for name, frame in (("primary", primary), ("fold_metrics", fold_metrics),
                    ("signer_metrics", signer_metrics), ("model_family", model_family),
                    ("secondary", secondary), ("secondary_inference", inference),
                    ("control_gap", control_gap), ("efficiency", efficiency),
                    ("class_summary", class_summary), ("class_recall", class_recall)):
    atomic_write(OUT[name], lambda h, f=frame: f.to_csv(h, index=False))
atomic_write(OUT["limitations"], lambda h: h.write(
    json.dumps({"limitations": LIMITATIONS, "deferred": DEFERRED}, indent=2).encode()))
atomic_write(OUT["contract"], lambda h: h.write(
    json.dumps(FINAL_CONTRACT, indent=2, sort_keys=True, default=str).encode()))
reloaded = json.loads(OUT["contract"].read_text(encoding="utf-8"))
require("the analysis contract id reproduces from the file on disk",
        sha16(json.dumps(reloaded, sort_keys=True, separators=(",", ":"), default=str))
        == FINAL_ANALYSIS_CONTRACT_ID, FINAL_ANALYSIS_CONTRACT_ID)

# ---- thesis-ready markdown -------------------------------------------------
lines: List[str] = []
lines.append("# Final results\n")
lines.append("## Primary — signer-independent recognition of unseen signers\n")
lines.append(f"Every prediction below comes from a model that never trained or validated "
             f"on that clip's signer. {N_CLIPS:,} out-of-fold predictions per "
             "configuration.\n")
lines.append(primary[["model", "representation", "accuracy", "macro_f1",
                      "signer_macro_accuracy", "fold_accuracy_mean", "fold_accuracy_sd",
                      "signer_min_accuracy", "signer_max_accuracy", "signer_spread"]]
             .round(4).to_markdown(index=False))
lines.append(f"\nBest observed configuration: **{BEST_CONFIGURATION['model']} "
             f"{BEST_CONFIGURATION['representation']}**, "
             f"{PRIMARY_METRIC} {BEST_CONFIGURATION[PRIMARY_METRIC]:.4f}. Identified from "
             "the recomputed values, not pre-designated.\n")
lines.append("## Secondary — Full vs Reduced representation\n")
lines.append(inference[["model", "statistic_T_reduced_minus_full",
                        "fold_difference_min", "fold_difference_max",
                        "folds_favouring_reduced", "bootstrap_ci_low",
                        "bootstrap_ci_high", "p_raw", "p_holm"]]
             .round(5).to_markdown(index=False))
lines.append("\nAccuracy is the only inferential metric. Effect sizes and the interval "
             "lead; the permutation p has a hard floor at n = 8. No equivalence claim.\n")
lines.append("## Secondary / diagnostic — signer-overlap performance gap\n")
lines.append(control_gap[["model", "representation", "signer_overlap_accuracy",
                          "signer_independent_oof_accuracy",
                          "signer_overlap_performance_gap"]].round(4).to_markdown(index=False))
lines.append("\nThe signer-overlap performance gap was substantially larger for the "
             "neural sequence models than for Random Forest. The overlap is intentional "
             "and is not leakage; contributors are unseparated.\n")
lines.append("## Secondary — efficiency\n")
lines.append(efficiency[["model", "mean_training_seconds_full",
                         "mean_training_seconds_reduced",
                         "training_time_ratio_full_over_reduced"]]
             .round(3).to_markdown(index=False))
lines.append("\nNo universal computational advantage is claimed; the saving is quoted "
             "per model family.\n")
lines.append("## Limitations\n")
for key, text in LIMITATIONS.items():
    lines.append(f"- **{key}** — {text}")
atomic_write(OUT["thesis_tables"], lambda h: h.write("\n".join(lines).encode()))

MANIFEST = {
    "final_analysis_contract_id": FINAL_ANALYSIS_CONTRACT_ID,
    "final_analysis_contract": FINAL_CONTRACT,
    "best_observed_configuration": BEST_CONFIGURATION,
    "best_observed_note": "identified from the recomputed values, not pre-designated",
    "verified_digests": {"out_of_fold": {f"{m}|{b}": REQUIRED_OOF_DIGEST[(m, b)]
                                         for m in MODELS for b in BRANCHES},
                         "control": {f"{m}|{b}": REQUIRED_CONTROL_DIGEST[(m, b)]
                                     for m in MODELS for b in BRANCHES}},
    "primary_results": primary.to_dict("records"),
    "secondary_inference": inference.to_dict("records"),
    "control_gap": control_gap.to_dict("records"),
    "efficiency": efficiency.to_dict("records"),
    "artefacts": {name: path.name for name, path in OUT.items()},
    "run_metadata": {"generated_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
                     "environment": ENVIRONMENT,
                     "resolved_paths": {m: str(MODEL_DIR[m]) for m in MODELS}
                     | {"notebook7": str(NB7_DIR)},
                     "runtime_seconds": round(time.time() - STARTED, 1),
                     "note": "excluded from final_analysis_contract_id"},
}
atomic_write(OUT["manifest"], lambda h: h.write(
    json.dumps(MANIFEST, indent=2, sort_keys=True, default=str).encode()))
reloaded_manifest = json.loads(OUT["manifest"].read_text(encoding="utf-8"))
require("the analysis contract id reproduces from the manifest on disk",
        sha16(json.dumps(reloaded_manifest["final_analysis_contract"], sort_keys=True,
                         separators=(",", ":"), default=str))
        == FINAL_ANALYSIS_CONTRACT_ID
        == reloaded_manifest["final_analysis_contract_id"], FINAL_ANALYSIS_CONTRACT_ID)
require("run metadata sits outside the hashed contract",
        "generated_at_utc" not in json.dumps(
            reloaded_manifest["final_analysis_contract"]), "separated")

# ---- report last ----------------------------------------------------------
FINAL_FAILURES = failures()
SKIPPED = [entry for entry in CHECKS if entry["status"] == "skip"]
VERDICT = "FAIL" if FINAL_FAILURES else "PASS"
report = {
    "notebook": "13-final-comparison-and-inference", "verdict": VERDICT,
    "final_analysis_contract_id": FINAL_ANALYSIS_CONTRACT_ID,
    "checks_total": len(CHECKS),
    "checks_passed": len([c for c in CHECKS if c["status"] == "pass"]),
    "checks_failed": len(FINAL_FAILURES), "checks_skipped": len(SKIPPED),
    "checks": CHECKS, "primary_results": primary.to_dict("records"),
    "secondary_inference": inference.to_dict("records"),
    "best_observed_configuration": BEST_CONFIGURATION,
    "runtime_seconds": round(time.time() - STARTED, 1),
    "run_metadata": MANIFEST["run_metadata"],
}
atomic_write(OUT["report_json"], lambda h: h.write(
    json.dumps(report, indent=2, default=str).encode()))
atomic_write(OUT["report_csv"], lambda h: pd.DataFrame(CHECKS)[
    ["check", "status", "passed", "detail"]].to_csv(h, index=False))
print(f"wrote {OUT['report_json'].name} and {OUT['report_csv'].name} "
      f"({len(CHECKS)} checks, verdict {VERDICT})")

if FINAL_FAILURES:
    for entry in FINAL_FAILURES:
        print(f"  FAIL  {entry['check']}: {entry['detail']}")
    raise RuntimeError(f"{len(FINAL_FAILURES)} checks failed. The report was written for "
                       "diagnosis, but this analysis is NOT usable.")

written = sorted({p for p in OUT.values() if p.exists()}
                 | set(WORKING.glob("final_confusion_*.npy")))
print("=" * 78)
print("NOTEBOOK 13 COMPLETE - FINAL ANALYSIS FROZEN")
print("=" * 78)
print(f"  final analysis contract id : {FINAL_ANALYSIS_CONTRACT_ID}")
print(f"  experiment id              : {REQUIRED_EXPERIMENT_ID}")
for model in MODELS:
    print(f"  {model:<7} model contract     : {MODEL_SPEC[model]['contract']}")
print()
print(f"  PRIMARY — signer-independent recognition of unseen signers "
      f"({N_CLIPS:,} predictions per configuration)")
for row in primary.itertuples():
    print(f"    {row.model:<7} {row.representation:<8} "
          f"accuracy {row.accuracy:.4f} | macro-F1 {row.macro_f1:.4f} | "
          f"signer-macro {row.signer_macro_accuracy:.4f} | "
          f"folds {row.fold_accuracy_mean:.4f} +/- {row.fold_accuracy_sd:.4f} | "
          f"signers {row.signer_min_accuracy:.4f}-{row.signer_max_accuracy:.4f}")
print(f"\n  Best observed: {BEST_CONFIGURATION['model']} "
      f"{BEST_CONFIGURATION['representation']}, {PRIMARY_METRIC} "
      f"{BEST_CONFIGURATION[PRIMARY_METRIC]:.4f} (from the recomputed values)")
print("\n  SECONDARY — Full vs Reduced (accuracy only, effect sizes first)")
for row in inference.itertuples():
    print(f"    {row.model:<7} T {row.statistic_T_reduced_minus_full:+.5f} | folds "
          f"{row.fold_difference_min:+.4f}..{row.fold_difference_max:+.4f} | "
          f"{row.folds_favouring_reduced}/{N_FOLDS} favour reduced | 95% CI "
          f"[{row.bootstrap_ci_low:+.4f}, {row.bootstrap_ci_high:+.4f}] | "
          f"p {row.p_raw:.7f} | Holm {row.p_holm:.7f}")
print("\n  SECONDARY / DIAGNOSTIC — signer-overlap performance gap")
for row in control_gap.itertuples():
    print(f"    {row.model:<7} {row.representation:<8} "
          f"{row.signer_overlap_performance_gap:+.4f}")
print("\n  SECONDARY — efficiency (per family, no universal claim)")
for row in efficiency.itertuples():
    print(f"    {row.model:<7} training time full/reduced "
          f"{row.training_time_ratio_full_over_reduced:.3f}x")
print(f"\n  checks   : {report['checks_passed']} passed, {report['checks_failed']} "
      f"failed, {report['checks_skipped']} skipped of {len(CHECKS)}")
print(f"  runtime  : {(time.time() - STARTED) / 60:.1f} min")
print(f"  artefacts: {len(written)} files")
print("\n  Signer-independent EVALUATION is established by the protocol. The stronger")
print("  claim that the representation contains no signer information is NOT made.")
print("  No cross-model significance test. No equivalence claim. Both deferred")
print("  sensitivities remain unrun and cannot replace any primary result.")
print("\n" + "=" * 78)
print(f"  Quote final analysis contract id {FINAL_ANALYSIS_CONTRACT_ID} in the thesis.")
print("=" * 78)